# 🌾 AgriVision — Deep Learning Training Pipeline (Google Colab)
Notebook ini melatih model klasifikasi daun tomat (**MobileNetV2**) menggunakan GPU T4 gratis di Google Colab, kemudian langsung mengunggah model terbaik (`best_model.pth`) ke **Hugging Face Hub** sebagai Model Registry terpusat.

### ⚡ Alur Kerja (Workflow):
1. **GPU Acceleration**: Pastikan Runtime disetel ke GPU (Runtime → Change runtime type → T4 GPU).
2. **Download & Filter Dataset**: Mengunduh dataset tomat (PlantVillage) dan memfilter 3 kelas target:
   - `Tomato___Bacterial_spot`
   - `Tomato___Late_blight`
   - `Tomato___healthy`
3. **Augmentasi Data**: Mensimulasikan variasi pencahayaan dan sudut foto di lapangan sawah.
4. **Transfer Learning**: Melatih MobileNetV2 dengan PyTorch.
5. **Push to Hugging Face**: Mengunggah model `best_model.pth` ke repository Hugging Face Anda.

## 1. Setup Environment & Fix Dependensi (Zero Conflict)

In [ ]:
# 1. Kunci Pillow ke versi stabil (<11.0.0) agar kompatibel sempurna dengan Torchvision di Colab
!pip install -q --force-reinstall "Pillow<11.0.0"

# 2. Install library eksternal yang dibutuhkan (huggingface-hub dipin <2.0 agar diffusers Colab tidak komplain)
!pip install -q "huggingface-hub<2.0" kagglehub

print("[OK] Dependensi sukses terpasang! Sekarang lakukan Runtime -> Restart session jika ini pertama kali.")

## 2. Unduh Dataset Tomat (PlantVillage)

In [ ]:
import os
import shutil
from pathlib import Path
import kagglehub

# 1. Download PlantVillage dataset menggunakan kagglehub (cepat & otomatis)
print("[INFO] Mengunduh dataset PlantVillage via kagglehub...")
raw_path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
print(f"[INFO] Dataset diunduh di: {raw_path}")

# 2. Siapkan direktori lokal untuk 3 kelas spesifik tomat kita
TARGET_DATA_DIR = "./data/raw/tomato"
CLASSES = ["Tomato___Bacterial_spot", "Tomato___Late_blight", "Tomato___healthy"]

for c in CLASSES:
    os.makedirs(os.path.join(TARGET_DATA_DIR, c), exist_ok=True)

# 3. Cari dan salin hanya 3 kelas tomat ke TARGET_DATA_DIR
print("[INFO] Memfilter 3 kelas daun tomat target...")
for root, dirs, files in os.walk(raw_path):
    for d in dirs:
        if d in CLASSES:
            src_dir = os.path.join(root, d)
            dest_dir = os.path.join(TARGET_DATA_DIR, d)
            for f in os.listdir(src_dir):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    shutil.copy(os.path.join(src_dir, f), os.path.join(dest_dir, f))

# Verifikasi jumlah gambar per kelas
print("\n--- Verifikasi Dataset Tomat ---")
for c in CLASSES:
    count = len(os.listdir(os.path.join(TARGET_DATA_DIR, c)))
    print(f"Kelas: {c} -> {count} gambar")

## 3. Data Augmentation & DataLoader PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

# Pipeline Augmentasi (mensimulasikan kondisi riil sawah: rotasi, intensitas cahaya, crop acak)
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(root=TARGET_DATA_DIR, transform=train_transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# Gunakan val_transform khusus untuk evaluasi validasi
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"[INFO] Total Sampel: {len(full_dataset)} (Train: {train_size}, Val: {val_size})")
print(f"[INFO] Urutan Kelas: {full_dataset.classes}")

## 4. Model Training (MobileNetV2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Perangkat Latih: {device}")

# 1. Arsitektur MobileNetV2 (Pre-trained di ImageNet)
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# 2. Freeze feature backbone
for param in model.parameters():
    param.requires_grad = False

# 3. Ganti classifier layer untuk 3 kelas
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(full_dataset.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=LEARNING_RATE)

best_acc = 0.0
BEST_MODEL_PATH = "best_model.pth"

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 20)
    
    # Training
    model.train()
    running_loss, running_corrects = 0.0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    train_loss = running_loss / len(train_loader.dataset)
    train_acc = running_corrects.double() / len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss, val_corrects = 0.0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_loader.dataset)
    val_acc = val_corrects.double() / len(val_loader.dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f">>> Model terbaik disimpan dengan Akurasi: {best_acc:.4f} <<<")

print(f"\n[COMPLETED] Pelatihan selesai! Akurasi Validasi Terbaik: {best_acc:.4f}")

## 5. Unggah Model Langsung ke Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, login

# Masukkan Token HF Write Anda saat menjalankan di Colab
HF_TOKEN = "MASUKKAN_HF_WRITE_TOKEN_ANDA_DISINI"
REPO_ID = "ginganomercy/agrivision-mobilenetv2"

login(token=HF_TOKEN)
api = HfApi()

# Buat repository publik jika belum ada
api.create_repo(repo_id=REPO_ID, private=False, exist_ok=True)
print(f"[INFO] Repository {REPO_ID} siap.")

# Upload file best_model.pth
print("[INFO] Mengunggah model ke Hugging Face Hub...")
api.upload_file(
    path_or_fileobj=BEST_MODEL_PATH,
    path_in_repo="best_model.pth",
    repo_id=REPO_ID,
    commit_message=f"Upload trained MobileNetV2 with validation accuracy {best_acc:.4f}"
)

print(f"\n[BERHASIL] Model telah live di: https://huggingface.co/{REPO_ID}")
print("Server Bare-Metal Docker Swarm Anda siap menarik model ini!")